## Bronze to Silver: Data Cleansing and Transformation

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

catalog_name = "ecommerce"

In [ ]:
broze_order = spark.table(f"{catalog_name}.broze.brz_order_items")

display(broze_order)

In [ ]:
# Tranformation: check duplicate id + item seq
silver_order = broze_order.groupBy(["order_id", "item_seq"]).count().where("count > 1")
display(silver_order)

# Drop duplicate
silver_order = broze_order.dropDuplicates(['order_id', 'item_seq'])

display(silver_order)

In [ ]:
# Transformation: remove % from discount 
silver_order = silver_order.withColumn("discount_pct", F.regexp_replace("discount_pct", "%", "").cast(IntegerType()))

display(silver_order)

In [ ]:
# check quantity..
silver_order.select('quantity').distinct().show()

silver_order = silver_order.withColumn('quantity', F.regexp_replace('quantity', 'Two', '2').cast(IntegerType()))

# Convert unit price, tax amount string, item squence to float..
silver_order = silver_order.withColumn('unit_price', F.regexp_replace('unit_price', '\\$', '').cast(FloatType()))\
                            .withColumn('tax_amount', F.regexp_replace('tax_amount', '\\$', '').cast(FloatType()))\
                            .withColumn('item_seq', F.col('item_seq').cast(IntegerType()))
 
display(silver_order)

In [ ]:
# convert string to date..
silver_order = silver_order.withColumn('dt', F.to_date('dt', 'yyyy-MM-dd'))\
                            .withColumn('order_ts',  F.coalesce(
                                        F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"),  # matches 2025-08-01 22:53:52
                                        F.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")      # fallback for 01-08-2025 22:53
    ))

display(silver_order)

In [ ]:
silver_order.printSchema()

In [ ]:
silver_order.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.silver.slv_orders")

In [ ]:
%%sql
SELECT * FROM ecommerce.silver.slv_orders;